# Heart Disease UCI Dataset – Exploratory Data Analysis

**MLOps Assignment 01 – BITS Pilani (AIMLCZG523)**

This notebook covers:
1. Dataset acquisition
2. Initial inspection and missing-value analysis
3. Univariate analysis (histograms)
4. Class distribution
5. Correlation heatmap
6. Feature–target relationship analysis
7. EDA summary

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))  # project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded')

## 1 – Dataset Acquisition

In [ ]:
import subprocess, pathlib

raw_path = pathlib.Path('../data/raw/heart_disease.csv')
if not raw_path.exists():
    subprocess.run(['python', '../data/download_data.py'], check=True)
    
df = pd.read_csv(raw_path)
df.replace('?', np.nan, inplace=True)
# Binarise target
df['target'] = (df['target'].astype(float) > 0).astype(int)
print(f'Shape: {df.shape}')
df.head()

## 2 – Initial Inspection & Missing Values

In [ ]:
print('=== dtypes ===')
print(df.dtypes)
print('\n=== describe ===')
df.describe(include='all')

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('No missing values found!')
else:
    print(missing_df)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(x=missing_df.index, y=missing_df['Missing %'], ax=ax)
    ax.set_title('Missing Values (%)')
    ax.set_ylabel('Percentage')
    plt.tight_layout()
    plt.show()

## 3 – Univariate Analysis (Histograms)

In [ ]:
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    if col in df.columns:
        df[col].astype(float).hist(ax=axes[i], bins=25, color='steelblue', edgecolor='white')
        axes[i].set_title(col)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')

# Hide empty subplot
axes[-1].set_visible(False)
fig.suptitle('Distribution of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/eda_histograms.png', bbox_inches='tight')
plt.show()

## 4 – Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['target'].value_counts()
labels = ['No Heart Disease', 'Heart Disease']
axes[0].bar(labels, [counts.get(0, 0), counts.get(1, 0)],
            color=['#2196F3', '#F44336'], edgecolor='white')
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Count')
for i, v in enumerate([counts.get(0, 0), counts.get(1, 0)]):
    axes[0].text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie([counts.get(0, 0), counts.get(1, 0)],
            labels=labels, autopct='%1.1f%%',
            colors=['#2196F3', '#F44336'], startangle=90)
axes[1].set_title('Class Distribution (%)')

fig.suptitle('Target Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/eda_class_distribution.png', bbox_inches='tight')
plt.show()
print(f'Class balance ratio: {counts.get(1,0)/counts.get(0,1):.3f}')

## 5 – Correlation Heatmap

In [ ]:
numeric_df = df[numerical_cols + ['target']].apply(pd.to_numeric, errors='coerce')
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (Numerical Features + Target)', fontsize=13)
plt.tight_layout()
plt.savefig('../screenshots/eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 6 – Feature–Target Relationship Analysis

In [ ]:
# Box plots: numerical features vs target
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    if col in df.columns:
        plot_df = df[[col, 'target']].copy()
        plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
        plot_df['Disease'] = plot_df['target'].map({0: 'No Disease', 1: 'Disease'})
        sns.boxplot(data=plot_df, x='Disease', y=col, ax=axes[i],
                    palette={'No Disease': '#2196F3', 'Disease': '#F44336'})
        axes[i].set_title(f'{col} vs Target')

axes[-1].set_visible(False)
fig.suptitle('Numerical Features vs Target (Box Plots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/eda_feature_target_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
# Categorical features vs target
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']
cat_cols = [c for c in cat_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols[:6]):
    plot_df = df[[col, 'target']].dropna()
    plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
    grouped = plot_df.groupby([col, 'target']).size().unstack(fill_value=0)
    grouped.plot(kind='bar', ax=axes[i], color=['#2196F3', '#F44336'],
                 edgecolor='white', legend=(i == 0))
    axes[i].set_title(f'{col} vs Target')
    axes[i].set_xlabel(col)
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=0)

fig.suptitle('Categorical Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/eda_categorical_features.png', bbox_inches='tight')
plt.show()

## 7 – Interactive Scatter Matrix (Plotly)

In [ ]:
scatter_df = df[numerical_cols + ['target']].apply(pd.to_numeric, errors='coerce').dropna()
scatter_df['Disease'] = scatter_df['target'].map({0: 'No Disease', 1: 'Disease'})

fig = px.scatter_matrix(
    scatter_df,
    dimensions=numerical_cols,
    color='Disease',
    color_discrete_map={'No Disease': '#2196F3', 'Disease': '#F44336'},
    title='Scatter Matrix of Numerical Features',
    opacity=0.6,
)
fig.update_traces(diagonal_visible=False)
fig.show()

## 8 – EDA Summary

| Finding | Detail |
|---------|--------|
| Dataset size | 303 rows, 14 columns |
| Missing values | Present in `ca` and `thal` columns (~1%) |
| Class balance | Approximately 54% disease, 46% no disease |
| Key correlators | `thalach` (−), `oldpeak` (+), `cp` (+), `exang` (+) |
| Age distribution | Skewed right; older patients at higher risk |
| Cholesterol | Weakly correlated with target |

**Preprocessing decisions:**
- Impute missing values with median (numerical) / mode (categorical)
- StandardScale numerical features
- OneHot-encode categorical features
- Stratified 80/20 train/test split